# Phase 1

- **1A.** Prepare notebook and raw files
- **1B.** Inspect CSV schemas
- **1C.** Clean PM2.5 data
- **1D.** Clean weather data
- **1E.** Join by hourly UTC timestamp
- **1F.** Validate and save the canonical dataset

### **1A.** Prepare notebook and raw files

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

POLLUTION_CSV_PATH = RAW_DATA_DIR / "openaq_pm25_hourly.csv"
WEATHER_CSV_PATH = RAW_DATA_DIR / "open_meteo_historical_weather.csv"

print("Project root:", PROJECT_ROOT)
print("Pollution file exists:", POLLUTION_CSV_PATH.exists())
print("Weather file exists:", WEATHER_CSV_PATH.exists())

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor
Pollution file exists: True
Weather file exists: True


In [2]:
print("pandas version:", pd.__version__)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)

pandas version: 3.0.5
Raw data directory: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/raw
Processed data directory: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/processed


### **1B.** Inspect CSV schemas

We need to confirm:

- File row counts
- Column names
- First few records
- Timestamp column names
- PM2.5 value column name
- Data types
- Missing values
- Duplicate rows at the raw-file level

In [3]:
pollution_raw = pd.read_csv(POLLUTION_CSV_PATH)
weather_raw = pd.read_csv(WEATHER_CSV_PATH)

print("Pollution shape:", pollution_raw.shape)
print("Weather shape:", weather_raw.shape)

Pollution shape: (8695, 13)
Weather shape: (9144, 20)


In [4]:
print("Pollution columns:")
for column in pollution_raw.columns:
    print(f"- {column}")

print("\nWeather columns:")
for column in weather_raw.columns:
    print(f"- {column}")

Pollution columns:
- sensor_id
- location_id
- location_name
- provider
- parameter
- units
- datetime_utc
- datetime_to_utc
- value
- minimum
- maximum
- median
- standard_deviation

Weather columns:
- datetime_utc
- temperature_2m
- relative_humidity_2m
- dew_point_2m
- surface_pressure
- precipitation
- rain
- cloud_cover
- visibility
- wind_speed_10m
- wind_direction_10m
- wind_gusts_10m
- source_latitude
- source_longitude
- source_elevation
- source_timezone
- utc_offset_seconds
- requested_start_date
- requested_end_date
- weather_source


In [5]:
print("Pollution sample:")
display(pollution_raw.head())

print("Weather sample:")
display(weather_raw.head())

Pollution sample:


,sensor_id,location_id,location_name,provider,parameter,units,datetime_utc,datetime_to_utc,value,minimum,maximum,median,standard_deviation
0,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2025-07-08 00:00:00+00:00,2025-07-08 01:00:00+00:00,13.202458,13.202458,13.202458,NaN,NaN
1,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2025-07-08 01:00:00+00:00,2025-07-08 02:00:00+00:00,14.268958,14.268958,14.268958,NaN,NaN
2,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2025-07-08 02:00:00+00:00,2025-07-08 03:00:00+00:00,15.587583,15.587583,15.587583,NaN,NaN
3,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2025-07-08 03:00:00+00:00,2025-07-08 04:00:00+00:00,14.713375,14.713375,14.713375,NaN,NaN
4,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2025-07-08 04:00:00+00:00,2025-07-08 05:00:00+00:00,17.668875,17.668875,17.668875,NaN,NaN


Weather sample:


,datetime_utc,temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation,rain,cloud_cover,visibility,wind_speed_10m,wind_direction_10m,wind_gusts_10m,source_latitude,source_longitude,source_elevation,source_timezone,utc_offset_seconds,requested_start_date,requested_end_date,weather_source
0,2025-07-08 00:00:00+00:00,28.5,87,26.2,997.3,0.1,0.1,49,NaN,3.6,225,8.6,24.850615,67.08915,5.0,GMT,0,2025-07-08,2025-07-31,open_meteo_archive
1,2025-07-08 01:00:00+00:00,29.1,82,25.8,997.9,0.0,0.0,98,NaN,1.5,263,5.4,24.850615,67.08915,5.0,GMT,0,2025-07-08,2025-07-31,open_meteo_archive
2,2025-07-08 02:00:00+00:00,29.4,80,25.6,998.5,0.0,0.0,82,NaN,0.7,270,5.0,24.850615,67.08915,5.0,GMT,0,2025-07-08,2025-07-31,open_meteo_archive
3,2025-07-08 03:00:00+00:00,30.2,75,25.3,998.9,0.0,0.0,79,NaN,2.2,228,10.1,24.850615,67.08915,5.0,GMT,0,2025-07-08,2025-07-31,open_meteo_archive
4,2025-07-08 04:00:00+00:00,31.1,70,24.9,999.3,0.0,0.0,32,NaN,3.2,199,13.7,24.850615,67.08915,5.0,GMT,0,2025-07-08,2025-07-31,open_meteo_archive


In [6]:
print("Last pollution rows:")
display(pollution_raw.tail())

print("Last weather rows:")
display(weather_raw.tail())

Last pollution rows:


,sensor_id,location_id,location_name,provider,parameter,units,datetime_utc,datetime_to_utc,value,minimum,maximum,median,standard_deviation
8690,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2026-07-23 19:00:00+00:00,2026-07-23 20:00:00+00:00,11.1,11.1,11.1,NaN,NaN
8691,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2026-07-23 20:00:00+00:00,2026-07-23 21:00:00+00:00,9.9,9.9,9.9,NaN,NaN
8692,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2026-07-23 21:00:00+00:00,2026-07-23 22:00:00+00:00,9.2,9.2,9.2,NaN,NaN
8693,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2026-07-23 22:00:00+00:00,2026-07-23 23:00:00+00:00,11.7,11.7,11.7,NaN,NaN
8694,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2026-07-23 23:00:00+00:00,2026-07-24 00:00:00+00:00,8.3,8.3,8.3,NaN,NaN


Last weather rows:


,datetime_utc,temperature_2m,relative_humidity_2m,dew_point_2m,surface_pressure,precipitation,rain,cloud_cover,visibility,wind_speed_10m,wind_direction_10m,wind_gusts_10m,source_latitude,source_longitude,source_elevation,source_timezone,utc_offset_seconds,requested_start_date,requested_end_date,weather_source
9139,2026-07-23 19:00:00+00:00,29.2,82,25.8,998.8,0.0,0.0,41,NaN,10.4,238,27.7,24.850615,67.08915,5.0,GMT,0,2026-07-01,2026-07-23,open_meteo_archive
9140,2026-07-23 20:00:00+00:00,28.9,83,25.7,998.6,0.0,0.0,43,NaN,8.8,242,25.2,24.850615,67.08915,5.0,GMT,0,2026-07-01,2026-07-23,open_meteo_archive
9141,2026-07-23 21:00:00+00:00,28.7,83,25.6,997.9,0.0,0.0,88,NaN,8.1,245,21.2,24.850615,67.08915,5.0,GMT,0,2026-07-01,2026-07-23,open_meteo_archive
9142,2026-07-23 22:00:00+00:00,28.7,82,25.4,997.8,0.0,0.0,98,NaN,7.0,249,20.2,24.850615,67.08915,5.0,GMT,0,2026-07-01,2026-07-23,open_meteo_archive
9143,2026-07-23 23:00:00+00:00,28.6,83,25.4,997.7,0.0,0.0,45,NaN,5.2,254,16.9,24.850615,67.08915,5.0,GMT,0,2026-07-01,2026-07-23,open_meteo_archive


In [7]:
print("Pollution data types:")
display(pollution_raw.dtypes.to_frame(name="dtype"))

print("Weather data types:")
display(weather_raw.dtypes.to_frame(name="dtype"))

Pollution data types:


,dtype
sensor_id,int64
location_id,int64
location_name,str
provider,str
parameter,str
units,str
datetime_utc,str
datetime_to_utc,str
value,float64
minimum,float64


Weather data types:


,dtype
datetime_utc,str
temperature_2m,float64
relative_humidity_2m,int64
dew_point_2m,float64
surface_pressure,float64
precipitation,float64
rain,float64
cloud_cover,int64
visibility,float64
wind_speed_10m,float64


In [8]:
pollution_missing = (
    pollution_raw.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="missing_count")
)

weather_missing = (
    weather_raw.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="missing_count")
)

print("Pollution missing values:")
display(pollution_missing)

print("Weather missing values:")
display(weather_missing)

Pollution missing values:


,missing_count
median,8695
standard_deviation,8695
location_name,0
location_id,0
sensor_id,0
parameter,0
provider,0
units,0
datetime_utc,0
value,0


Weather missing values:


,missing_count
visibility,9144
datetime_utc,0
relative_humidity_2m,0
temperature_2m,0
dew_point_2m,0
surface_pressure,0
rain,0
precipitation,0
cloud_cover,0
wind_speed_10m,0


In [9]:
print(
    "Exact duplicate pollution rows:",
    pollution_raw.duplicated().sum(),
)

print(
    "Exact duplicate weather rows:",
    weather_raw.duplicated().sum(),
)

Exact duplicate pollution rows: 0
Exact duplicate weather rows: 0


In [10]:
likely_metadata_columns = [
    "parameter",
    "pollutant",
    "unit",
    "sensor_id",
    "sensors_id",
    "location_id",
    "locations_id",
]

for column in likely_metadata_columns:
    if column in pollution_raw.columns:
        print(f"\nUnique values in '{column}':")
        print(pollution_raw[column].dropna().unique()[:20])


Unique values in 'parameter':
<ArrowStringArray>
['pm25']
Length: 1, dtype: str

Unique values in 'sensor_id':
[13387396]

Unique values in 'location_id':
[4814327]


### **1C.** Cleaning the pollution data

For the first part of Phase 1C, we will:

- Select required columns.
- Parse datetime_utc.
- Confirm timezone-aware UTC timestamps.
- Sort chronologically.
- Check duplicate timestamps.

In [11]:
pollution = pollution_raw[
    [
        "sensor_id",
        "location_id",
        "location_name",
        "provider",
        "parameter",
        "units",
        "datetime_utc",
        "value",
    ]
].copy()

print("Selected pollution shape:", pollution.shape)
print("Selected pollution columns:")
print(pollution.columns.tolist())

Selected pollution shape: (8695, 8)
Selected pollution columns:
['sensor_id', 'location_id', 'location_name', 'provider', 'parameter', 'units', 'datetime_utc', 'value']


In [12]:
pollution["datetime_utc"] = pd.to_datetime(
    pollution["datetime_utc"],
    errors="coerce",
    utc=True,
)

print("Timestamp dtype:", pollution["datetime_utc"].dtype)
print(
    "Invalid timestamps:",
    pollution["datetime_utc"].isna().sum(),
)

Timestamp dtype: datetime64[us, UTC]
Invalid timestamps: 0


In [13]:
pollution = (
    pollution
    .sort_values("datetime_utc")
    .reset_index(drop=True)
)

print("First timestamp:", pollution["datetime_utc"].min())
print("Last timestamp:", pollution["datetime_utc"].max())
print(
    "Chronologically sorted:",
    pollution["datetime_utc"].is_monotonic_increasing,
)

First timestamp: 2025-07-08 00:00:00+00:00
Last timestamp: 2026-07-23 23:00:00+00:00
Chronologically sorted: True


In [14]:
duplicate_timestamp_count = pollution[
    "datetime_utc"
].duplicated().sum()

print(
    "Duplicate pollution timestamps:",
    duplicate_timestamp_count,
)

if duplicate_timestamp_count > 0:
    display(
        pollution[
            pollution["datetime_utc"].duplicated(
                keep=False
            )
        ].sort_values("datetime_utc")
    )

Duplicate pollution timestamps: 0


For the Second part of Phase 1C, we will:

- Ensure value is numeric.
- Inspect missing, zero, negative, minimum, and maximum values.
- Flag exact zeros and negative values.
- Replace them with missing values.
- Rename the cleaned column to pm25_ug_m3.

In [16]:
pollution["value"] = pd.to_numeric(
    pollution["value"],
    errors="coerce",
)

print("PM2.5 dtype:", pollution["value"].dtype)
print(
    "Missing or non-numeric PM2.5 values:",
    pollution["value"].isna().sum(),
)

PM2.5 dtype: float64
Missing or non-numeric PM2.5 values: 0


In [17]:
pm25_summary = {
    "row_count": len(pollution),
    "missing_values": int(pollution["value"].isna().sum()),
    "exact_zero_values": int(pollution["value"].eq(0).sum()),
    "negative_values": int(pollution["value"].lt(0).sum()),
    "minimum_value": pollution["value"].min(),
    "maximum_value": pollution["value"].max(),
    "mean_value": pollution["value"].mean(),
    "median_value": pollution["value"].median(),
}

pm25_summary

{'row_count': 8695,
 'missing_values': 0,
 'exact_zero_values': 5,
 'negative_values': 0,
 'minimum_value': np.float64(0.0),
 'maximum_value': np.float64(533.1198354085287),
 'mean_value': np.float64(35.70608852510342),
 'median_value': np.float64(20.020458221435547)}

In [18]:
suspicious_pm25 = pollution.loc[
    pollution["value"].isna()
    | pollution["value"].eq(0)
    | pollution["value"].lt(0),
    [
        "datetime_utc",
        "value",
        "sensor_id",
        "location_id",
    ],
]

print("Suspicious PM2.5 records:", len(suspicious_pm25))
display(suspicious_pm25)

Suspicious PM2.5 records: 5


,datetime_utc,value,sensor_id,location_id
6198,2026-03-28 03:00:00+00:00,0.0,13387396,4814327
6200,2026-03-28 05:00:00+00:00,0.0,13387396,4814327
6206,2026-03-28 11:00:00+00:00,0.0,13387396,4814327
6207,2026-03-28 12:00:00+00:00,0.0,13387396,4814327
6229,2026-03-29 10:00:00+00:00,0.0,13387396,4814327


In [19]:
pollution["pm25_zero_flag"] = pollution["value"].eq(0)
pollution["pm25_negative_flag"] = pollution["value"].lt(0)

pollution["pm25_ug_m3"] = pollution["value"].mask(
    pollution["pm25_zero_flag"]
    | pollution["pm25_negative_flag"]
)

In [20]:
print(
    "Zero readings flagged:",
    pollution["pm25_zero_flag"].sum(),
)

print(
    "Negative readings flagged:",
    pollution["pm25_negative_flag"].sum(),
)

print(
    "Missing PM2.5 after cleaning:",
    pollution["pm25_ug_m3"].isna().sum(),
)

print(
    "Remaining exact zeros:",
    pollution["pm25_ug_m3"].eq(0).sum(),
)

print(
    "Remaining negative values:",
    pollution["pm25_ug_m3"].lt(0).sum(),
)

print(
    "Maximum PM2.5 retained:",
    pollution["pm25_ug_m3"].max(),
)

Zero readings flagged: 5
Negative readings flagged: 0
Missing PM2.5 after cleaning: 5
Remaining exact zeros: 0
Remaining negative values: 0
Maximum PM2.5 retained: 533.1198354085287


In [21]:
pollution_clean = pollution[
    [
        "datetime_utc",
        "pm25_ug_m3",
        "pm25_zero_flag",
        "pm25_negative_flag",
        "sensor_id",
        "location_id",
        "location_name",
        "provider",
    ]
].copy()

print("Clean pollution shape:", pollution_clean.shape)
display(pollution_clean.head())

Clean pollution shape: (8695, 8)


,datetime_utc,pm25_ug_m3,pm25_zero_flag,pm25_negative_flag,sensor_id,location_id,location_name,provider
0,2025-07-08 00:00:00+00:00,13.202458,False,False,13387396,4814327,Zafar Memon DHA,AirGradient
1,2025-07-08 01:00:00+00:00,14.268958,False,False,13387396,4814327,Zafar Memon DHA,AirGradient
2,2025-07-08 02:00:00+00:00,15.587583,False,False,13387396,4814327,Zafar Memon DHA,AirGradient
3,2025-07-08 03:00:00+00:00,14.713375,False,False,13387396,4814327,Zafar Memon DHA,AirGradient
4,2025-07-08 04:00:00+00:00,17.668875,False,False,13387396,4814327,Zafar Memon DHA,AirGradient


For the Third part of Phase 1C, we will Build the complete PM2.5 hourly timeline

In [22]:
# Create the expected hourly UTC timeline
EXPECTED_START_UTC = pd.Timestamp(
    "2025-07-08 00:00:00",
    tz="UTC",
)

EXPECTED_END_UTC = pd.Timestamp(
    "2026-07-23 23:00:00",
    tz="UTC",
)

expected_timeline = pd.DataFrame(
    {
        "datetime_utc": pd.date_range(
            start=EXPECTED_START_UTC,
            end=EXPECTED_END_UTC,
            freq="h",
        )
    }
)

print("Expected timeline rows:", len(expected_timeline))
print("First expected timestamp:", expected_timeline["datetime_utc"].min())
print("Last expected timestamp:", expected_timeline["datetime_utc"].max())

Expected timeline rows: 9144
First expected timestamp: 2025-07-08 00:00:00+00:00
Last expected timestamp: 2026-07-23 23:00:00+00:00


In [23]:
# Join cleaned PM2.5 onto the complete timeline
pollution_hourly = expected_timeline.merge(
    pollution_clean,
    on="datetime_utc",
    how="left",
    validate="one_to_one",
)

print("Complete pollution timeline shape:", pollution_hourly.shape)
print(
    "Missing PM2.5 hours:",
    pollution_hourly["pm25_ug_m3"].isna().sum(),
)

Complete pollution timeline shape: (9144, 8)
Missing PM2.5 hours: 454


In [24]:
# Calculate cleaned coverage
total_hours = len(pollution_hourly)

available_pm25_hours = (
    pollution_hourly["pm25_ug_m3"]
    .notna()
    .sum()
)

missing_pm25_hours = (
    pollution_hourly["pm25_ug_m3"]
    .isna()
    .sum()
)

coverage_percent = (
    available_pm25_hours / total_hours
) * 100

print("Total expected hours:", total_hours)
print("Available cleaned PM2.5 hours:", available_pm25_hours)
print("Missing cleaned PM2.5 hours:", missing_pm25_hours)
print(f"Cleaned PM2.5 coverage: {coverage_percent:.2f}%")

Total expected hours: 9144
Available cleaned PM2.5 hours: 8690
Missing cleaned PM2.5 hours: 454
Cleaned PM2.5 coverage: 95.03%


In [25]:
# Identify consecutive missing gaps
missing_mask = pollution_hourly[
    "pm25_ug_m3"
].isna()

gap_group = missing_mask.ne(
    missing_mask.shift()
).cumsum()

missing_gaps = (
    pollution_hourly.loc[
        missing_mask,
        ["datetime_utc"]
    ]
    .assign(gap_group=gap_group[missing_mask].to_numpy())
    .groupby("gap_group")
    .agg(
        gap_start_utc=("datetime_utc", "min"),
        gap_end_utc=("datetime_utc", "max"),
        missing_hours=("datetime_utc", "size"),
    )
    .reset_index(drop=True)
    .sort_values(
        "missing_hours",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("Number of missing gaps:", len(missing_gaps))
print(
    "Longest missing gap:",
    missing_gaps["missing_hours"].max(),
    "hours",
)

display(missing_gaps.head(15))

Number of missing gaps: 56
Longest missing gap: 274 hours


,gap_start_utc,gap_end_utc,missing_hours
0,2026-04-03 09:00:00+00:00,2026-04-14 18:00:00+00:00,274
1,2026-01-21 22:00:00+00:00,2026-01-22 14:00:00+00:00,17
2,2026-07-13 14:00:00+00:00,2026-07-14 02:00:00+00:00,13
3,2026-05-31 03:00:00+00:00,2026-05-31 14:00:00+00:00,12
4,2026-05-09 07:00:00+00:00,2026-05-09 18:00:00+00:00,12
5,2025-10-04 19:00:00+00:00,2025-10-05 04:00:00+00:00,10
6,2025-11-14 05:00:00+00:00,2025-11-14 12:00:00+00:00,8
7,2025-10-20 12:00:00+00:00,2025-10-20 19:00:00+00:00,8
8,2025-11-10 23:00:00+00:00,2025-11-11 05:00:00+00:00,7
9,2025-09-28 08:00:00+00:00,2025-09-28 14:00:00+00:00,7


In [26]:
assert len(pollution_hourly) == 9144
assert pollution_hourly["datetime_utc"].duplicated().sum() == 0
assert pollution_hourly["datetime_utc"].is_monotonic_increasing
assert pollution_hourly["pm25_ug_m3"].eq(0).sum() == 0
assert pollution_hourly["pm25_ug_m3"].lt(0).sum() == 0

print("Pollution cleaning validation passed.")

Pollution cleaning validation passed.


### **1D.** Cleaning the historical weather data

For the first part of Phase 1D, we will prepare weather timestamps and columns

In [27]:
# Define the required weather variables
WEATHER_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "surface_pressure",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
]

In [28]:
# Confirm all required columns exist
required_weather_columns = [
    "datetime_utc",
    *WEATHER_COLUMNS,
]

missing_required_columns = [
    column
    for column in required_weather_columns
    if column not in weather_raw.columns
]

print("Missing required weather columns:", missing_required_columns)

assert not missing_required_columns, (
    f"Required weather columns are missing: {missing_required_columns}"
)

Missing required weather columns: []


In [29]:
# Create the working weather DataFrame
weather = weather_raw[
    required_weather_columns
].copy()

print("Selected weather shape:", weather.shape)
print("Selected weather columns:")
print(weather.columns.tolist())

Selected weather shape: (9144, 11)
Selected weather columns:
['datetime_utc', 'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'surface_pressure', 'precipitation', 'rain', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m']


In [30]:
# Parse the weather timestamp as UTC
weather["datetime_utc"] = pd.to_datetime(
    weather["datetime_utc"],
    errors="coerce",
    utc=True,
)

print("Timestamp dtype:", weather["datetime_utc"].dtype)
print(
    "Invalid weather timestamps:",
    weather["datetime_utc"].isna().sum(),
)

Timestamp dtype: datetime64[us, UTC]
Invalid weather timestamps: 0


In [31]:
# Sort and inspect the historical range
weather = (
    weather
    .sort_values("datetime_utc")
    .reset_index(drop=True)
)

print("First weather timestamp:", weather["datetime_utc"].min())
print("Last weather timestamp:", weather["datetime_utc"].max())

print(
    "Chronologically sorted:",
    weather["datetime_utc"].is_monotonic_increasing,
)

First weather timestamp: 2025-07-08 00:00:00+00:00
Last weather timestamp: 2026-07-23 23:00:00+00:00
Chronologically sorted: True


In [32]:
# Validate duplicate timestamps
duplicate_weather_timestamps = (
    weather["datetime_utc"]
    .duplicated()
    .sum()
)

print(
    "Duplicate weather timestamps:",
    duplicate_weather_timestamps,
)

if duplicate_weather_timestamps > 0:
    display(
        weather.loc[
            weather["datetime_utc"].duplicated(keep=False)
        ].sort_values("datetime_utc")
    )

Duplicate weather timestamps: 0


In [33]:
# Validate the expected timeline
expected_weather_timestamps = pd.date_range(
    start=EXPECTED_START_UTC,
    end=EXPECTED_END_UTC,
    freq="h",
)

missing_weather_timestamps = (
    expected_weather_timestamps
    .difference(weather["datetime_utc"])
)

unexpected_weather_timestamps = (
    pd.DatetimeIndex(weather["datetime_utc"])
    .difference(expected_weather_timestamps)
)

print("Expected weather rows:", len(expected_weather_timestamps))
print("Actual weather rows:", len(weather))
print("Missing timestamps:", len(missing_weather_timestamps))
print("Unexpected timestamps:", len(unexpected_weather_timestamps))

Expected weather rows: 9144
Actual weather rows: 9144
Missing timestamps: 0
Unexpected timestamps: 0


For the Second part of Phase 1D, we will Validate weather values 

In [34]:
# Convert weather columns to numeric
for column in WEATHER_COLUMNS:
    weather[column] = pd.to_numeric(
        weather[column],
        errors="coerce",
    )

print("Weather data types:")
display(
    weather[WEATHER_COLUMNS]
    .dtypes
    .astype(str)
    .to_frame(name="dtype")
)

Weather data types:


,dtype
temperature_2m,float64
relative_humidity_2m,int64
dew_point_2m,float64
surface_pressure,float64
precipitation,float64
rain,float64
cloud_cover,int64
wind_speed_10m,float64
wind_direction_10m,int64
wind_gusts_10m,float64


In [35]:
# Check missing values
weather_missing_summary = (
    weather[WEATHER_COLUMNS]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="missing_count")
)

display(weather_missing_summary)

print(
    "Total missing weather values:",
    weather_missing_summary["missing_count"].sum(),
)

,missing_count
temperature_2m,0
relative_humidity_2m,0
dew_point_2m,0
surface_pressure,0
precipitation,0
rain,0
cloud_cover,0
wind_speed_10m,0
wind_direction_10m,0
wind_gusts_10m,0


Total missing weather values: 0


In [36]:
# Check infinite values
import numpy as np

weather_infinite_summary = pd.Series(
    {
        column: np.isinf(weather[column]).sum()
        for column in WEATHER_COLUMNS
    },
    name="infinite_count",
).sort_values(ascending=False)

display(weather_infinite_summary.to_frame())

print(
    "Total infinite weather values:",
    weather_infinite_summary.sum(),
)

,infinite_count
temperature_2m,0
relative_humidity_2m,0
dew_point_2m,0
surface_pressure,0
precipitation,0
rain,0
cloud_cover,0
wind_speed_10m,0
wind_direction_10m,0
wind_gusts_10m,0


Total infinite weather values: 0


In [37]:
# Inspect minimum and maximum values
weather_range_summary = weather[
    WEATHER_COLUMNS
].agg(["min", "max"]).T

weather_range_summary.columns = [
    "minimum",
    "maximum",
]

display(weather_range_summary)

,minimum,maximum
temperature_2m,9.2,42.1
relative_humidity_2m,6.0,100.0
dew_point_2m,-7.4,27.7
surface_pressure,993.6,1024.6
precipitation,0.0,20.0
rain,0.0,20.0
cloud_cover,0.0,100.0
wind_speed_10m,0.0,24.3
wind_direction_10m,1.0,360.0
wind_gusts_10m,1.1,59.8


In [38]:
# basic validity checks
weather_quality_checks = {
    "humidity_below_0": int(
        weather["relative_humidity_2m"].lt(0).sum()
    ),
    "humidity_above_100": int(
        weather["relative_humidity_2m"].gt(100).sum()
    ),
    "cloud_cover_below_0": int(
        weather["cloud_cover"].lt(0).sum()
    ),
    "cloud_cover_above_100": int(
        weather["cloud_cover"].gt(100).sum()
    ),
    "negative_precipitation": int(
        weather["precipitation"].lt(0).sum()
    ),
    "negative_rain": int(
        weather["rain"].lt(0).sum()
    ),
    "negative_wind_speed": int(
        weather["wind_speed_10m"].lt(0).sum()
    ),
    "negative_wind_gusts": int(
        weather["wind_gusts_10m"].lt(0).sum()
    ),
    "wind_direction_below_0": int(
        weather["wind_direction_10m"].lt(0).sum()
    ),
    "wind_direction_above_360": int(
        weather["wind_direction_10m"].gt(360).sum()
    ),
    "non_positive_pressure": int(
        weather["surface_pressure"].le(0).sum()
    ),
}

weather_quality_checks

{'humidity_below_0': 0,
 'humidity_above_100': 0,
 'cloud_cover_below_0': 0,
 'cloud_cover_above_100': 0,
 'negative_precipitation': 0,
 'negative_rain': 0,
 'negative_wind_speed': 0,
 'negative_wind_gusts': 0,
 'wind_direction_below_0': 0,
 'wind_direction_above_360': 0,
 'non_positive_pressure': 0}

In [39]:
# Final weather validation
assert len(weather) == 9144
assert weather["datetime_utc"].duplicated().sum() == 0
assert weather["datetime_utc"].is_monotonic_increasing

assert (
    weather[WEATHER_COLUMNS]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    sum(weather_quality_checks.values())
    == 0
)

print("Historical weather cleaning validation passed.")

Historical weather cleaning validation passed.


### **1E.** Joining the cleaned pollution and weather data.

In [40]:
# Prepare the clean weather DataFrame
weather_clean = weather[
    [
        "datetime_utc",
        *WEATHER_COLUMNS,
    ]
].copy()

print("Clean weather shape:", weather_clean.shape)

Clean weather shape: (9144, 11)


In [41]:
# Join pollution and weather
canonical_hourly = pollution_hourly.merge(
    weather_clean,
    on="datetime_utc",
    how="left",
    validate="one_to_one",
)

print("Canonical dataset shape:", canonical_hourly.shape)

Canonical dataset shape: (9144, 18)


In [42]:
# Inspect the joined columns
print("Canonical dataset columns:")

for column in canonical_hourly.columns:
    print(f"- {column}")

Canonical dataset columns:
- datetime_utc
- pm25_ug_m3
- pm25_zero_flag
- pm25_negative_flag
- sensor_id
- location_id
- location_name
- provider
- temperature_2m
- relative_humidity_2m
- dew_point_2m
- surface_pressure
- precipitation
- rain
- cloud_cover
- wind_speed_10m
- wind_direction_10m
- wind_gusts_10m


In [43]:
# Validate joined row counts and missing values
print("Total rows:", len(canonical_hourly))

print(
    "Duplicate timestamps:",
    canonical_hourly["datetime_utc"].duplicated().sum(),
)

print(
    "Missing PM2.5 values:",
    canonical_hourly["pm25_ug_m3"].isna().sum(),
)

print(
    "Total missing weather values:",
    canonical_hourly[WEATHER_COLUMNS]
    .isna()
    .sum()
    .sum(),
)

Total rows: 9144
Duplicate timestamps: 0
Missing PM2.5 values: 454
Total missing weather values: 0


In [44]:
# Inspect rows with missing PM2.5
missing_pm25_sample = canonical_hourly.loc[
    canonical_hourly["pm25_ug_m3"].isna(),
    [
        "datetime_utc",
        "pm25_ug_m3",
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
    ],
]

print("Rows with missing PM2.5:", len(missing_pm25_sample))

display(missing_pm25_sample.head(10))

Rows with missing PM2.5: 454


,datetime_utc,pm25_ug_m3,temperature_2m,relative_humidity_2m,wind_speed_10m
488,2025-07-28 08:00:00+00:00,NaN,31.4,64,20.9
1307,2025-08-31 11:00:00+00:00,NaN,30.4,65,17.6
1308,2025-08-31 12:00:00+00:00,NaN,29.6,70,17.7
1371,2025-09-03 03:00:00+00:00,NaN,27.9,75,9.4
1372,2025-09-03 04:00:00+00:00,NaN,29.2,66,8.7
1373,2025-09-03 05:00:00+00:00,NaN,30.4,60,9.3
1374,2025-09-03 06:00:00+00:00,NaN,31.3,57,11.2
1498,2025-09-08 10:00:00+00:00,NaN,27.9,82,16.2
1499,2025-09-08 11:00:00+00:00,NaN,28.1,79,17.1
1500,2025-09-08 12:00:00+00:00,NaN,27.7,83,13.9


In [45]:
# Validate timestamp alignment
assert canonical_hourly["datetime_utc"].equals(
    expected_timeline["datetime_utc"]
)

print("Timestamp alignment validation passed.")

Timestamp alignment validation passed.


In [46]:
# Final Phase 1E checks
assert len(canonical_hourly) == 9144

assert (
    canonical_hourly["datetime_utc"]
    .duplicated()
    .sum()
    == 0
)

assert canonical_hourly[
    "datetime_utc"
].is_monotonic_increasing

assert (
    canonical_hourly["pm25_ug_m3"]
    .isna()
    .sum()
    == 454
)

assert (
    canonical_hourly[WEATHER_COLUMNS]
    .isna()
    .sum()
    .sum()
    == 0
)

print("Canonical PM2.5-weather join validation passed.")

Canonical PM2.5-weather join validation passed.


### **1F.** finalize the canonical dataset.

For the first part of Phase 1F, we will finalize metadata and quality flags

In [47]:
# Inspect missing values in every column
canonical_missing_summary = (
    canonical_hourly
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="missing_count")
)

display(canonical_missing_summary)

,missing_count
pm25_ug_m3,454
pm25_zero_flag,449
location_name,449
pm25_negative_flag,449
sensor_id,449
location_id,449
provider,449
datetime_utc,0
temperature_2m,0
relative_humidity_2m,0


In [48]:
# Fill constant location and sensor metadata
canonical_hourly["sensor_id"] = (
    canonical_hourly["sensor_id"]
    .fillna(13387396)
    .astype("int64")
)

canonical_hourly["location_id"] = (
    canonical_hourly["location_id"]
    .fillna(4814327)
    .astype("int64")
)

canonical_hourly["location_name"] = (
    canonical_hourly["location_name"]
    .fillna("Zafar Memon DHA")
)

canonical_hourly["provider"] = (
    canonical_hourly["provider"]
    .fillna("AirGradient")
)

In [49]:
# Finalize PM2.5 quality flags
canonical_hourly["pm25_zero_flag"] = (
    canonical_hourly["pm25_zero_flag"]
    .fillna(False)
    .astype(bool)
)

canonical_hourly["pm25_negative_flag"] = (
    canonical_hourly["pm25_negative_flag"]
    .fillna(False)
    .astype(bool)
)

canonical_hourly["pm25_missing"] = (
    canonical_hourly["pm25_ug_m3"]
    .isna()
)

In [50]:
# Validate the flags
print(
    "Total missing PM2.5:",
    canonical_hourly["pm25_missing"].sum(),
)

print(
    "Zero readings flagged:",
    canonical_hourly["pm25_zero_flag"].sum(),
)

print(
    "Negative readings flagged:",
    canonical_hourly["pm25_negative_flag"].sum(),
)

print(
    "Missing metadata values:",
    canonical_hourly[
        [
            "sensor_id",
            "location_id",
            "location_name",
            "provider",
        ]
    ]
    .isna()
    .sum()
    .sum(),
)

Total missing PM2.5: 454
Zero readings flagged: 5
Negative readings flagged: 0
Missing metadata values: 0


In [51]:
# Verify the relationship between flags and missing values
assert canonical_hourly["pm25_missing"].sum() == 454
assert canonical_hourly["pm25_zero_flag"].sum() == 5
assert canonical_hourly["pm25_negative_flag"].sum() == 0

assert (
    canonical_hourly.loc[
        canonical_hourly["pm25_zero_flag"],
        "pm25_ug_m3",
    ]
    .isna()
    .all()
)

assert (
    canonical_hourly[
        [
            "sensor_id",
            "location_id",
            "location_name",
            "provider",
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

print("Canonical metadata and PM2.5 flags validated.")

Canonical metadata and PM2.5 flags validated.


For the Second part of Phase 1F, we will final the schema and save it

In [52]:
# Arrange the final column order
final_columns = [
    "datetime_utc",
    "sensor_id",
    "location_id",
    "location_name",
    "provider",
    "pm25_ug_m3",
    "pm25_missing",
    "pm25_zero_flag",
    "pm25_negative_flag",
    *WEATHER_COLUMNS,
]

canonical_hourly = canonical_hourly[
    final_columns
].copy()

print("Final canonical shape:", canonical_hourly.shape)
print("Final canonical columns:")

for column in canonical_hourly.columns:
    print(f"- {column}")

Final canonical shape: (9144, 19)
Final canonical columns:
- datetime_utc
- sensor_id
- location_id
- location_name
- provider
- pm25_ug_m3
- pm25_missing
- pm25_zero_flag
- pm25_negative_flag
- temperature_2m
- relative_humidity_2m
- dew_point_2m
- surface_pressure
- precipitation
- rain
- cloud_cover
- wind_speed_10m
- wind_direction_10m
- wind_gusts_10m


In [53]:
# Inspect final data types
final_dtypes = (
    canonical_hourly
    .dtypes
    .astype(str)
    .to_frame(name="dtype")
)

display(final_dtypes)

,dtype
datetime_utc,"datetime64[us, UTC]"
sensor_id,int64
location_id,int64
location_name,str
provider,str
pm25_ug_m3,float64
pm25_missing,bool
pm25_zero_flag,bool
pm25_negative_flag,bool
temperature_2m,float64


In [54]:
# Final complete validation
assert canonical_hourly.shape == (9144, 19)

assert canonical_hourly["datetime_utc"].duplicated().sum() == 0
assert canonical_hourly["datetime_utc"].is_monotonic_increasing

assert canonical_hourly["pm25_ug_m3"].isna().sum() == 454
assert canonical_hourly["pm25_missing"].sum() == 454

assert canonical_hourly["pm25_zero_flag"].sum() == 5
assert canonical_hourly["pm25_negative_flag"].sum() == 0

assert (
    canonical_hourly[WEATHER_COLUMNS]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    canonical_hourly[
        [
            "sensor_id",
            "location_id",
            "location_name",
            "provider",
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

print("Final canonical dataset validation passed.")

Final canonical dataset validation passed.


In [55]:
# Save as Parquet and CSV
PARQUET_OUTPUT_PATH = (
    PROCESSED_DATA_DIR
    / "pearls_aqi_canonical_hourly.parquet"
)

CSV_OUTPUT_PATH = (
    PROCESSED_DATA_DIR
    / "pearls_aqi_canonical_hourly.csv"
)

canonical_hourly.to_parquet(
    PARQUET_OUTPUT_PATH,
    index=False,
)

canonical_hourly.to_csv(
    CSV_OUTPUT_PATH,
    index=False,
)

print("Saved Parquet:", PARQUET_OUTPUT_PATH)
print("Saved CSV:", CSV_OUTPUT_PATH)

Saved Parquet: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/processed/pearls_aqi_canonical_hourly.parquet
Saved CSV: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/processed/pearls_aqi_canonical_hourly.csv


In [56]:
# Reload and verify the Parquet file
canonical_reloaded = pd.read_parquet(
    PARQUET_OUTPUT_PATH
)

print("Reloaded shape:", canonical_reloaded.shape)
print(
    "Reloaded timestamp dtype:",
    canonical_reloaded["datetime_utc"].dtype,
)

pd.testing.assert_frame_equal(
    canonical_hourly.reset_index(drop=True),
    canonical_reloaded.reset_index(drop=True),
    check_dtype=True,
)

print("Parquet reload validation passed.")

Reloaded shape: (9144, 19)
Reloaded timestamp dtype: datetime64[us, UTC]
Parquet reload validation passed.


In [57]:
# Confirm saved files exist
print(
    "Parquet exists:",
    PARQUET_OUTPUT_PATH.exists(),
)

print(
    "CSV exists:",
    CSV_OUTPUT_PATH.exists(),
)

print(
    "Parquet size:",
    PARQUET_OUTPUT_PATH.stat().st_size,
    "bytes",
)

print(
    "CSV size:",
    CSV_OUTPUT_PATH.stat().st_size,
    "bytes",
)

Parquet exists: True
CSV exists: True
Parquet size: 228219 bytes
CSV size: 1326897 bytes
